# 0. Start here — what `beamfeat` does, and how well

`beamfeat` searches for short algebraic formulas that explain a target, then tests
each candidate on data the search never saw. You get a readable equation plus a
q-value saying how much confidence it carries.

This series applies it to eleven problems across geometry, engineering, physics,
chemistry and biology. Every number below is produced by a notebook in this folder,
and every notebook runs end to end.

## The record

**It recovered the correct law in eight of eleven.** Where the target is a product,
ratio or power of the inputs, it finds the form and the fitted coefficient is the
physical constant:

| Notebook | Law recovered | Constant | Error |
|---|---|---|---|
| 01 | $\text{area} = \pi r^2$ | $\pi$ | 2% |
| 01 | $\text{perimeter} = 2\pi r$ | $2\pi$ | 2% |
| 02 | carat $\propto$ volume | fill fraction 35% | matches cut geometry |
| 06 | $r = k[\mathrm{H}][\mathrm{O_2}]$ | rate constant | **0.27%** |
| 06 | $r = k[\mathrm{OH}]^2$ | rate constant | **0.20%** |
| 07 | $\mathrm{Re} = \rho v D/\mu$ | — | exact reconstruction |
| 07 | Dittus–Boelter exponents | 0.8 and 0.4 | 0.07% / 0.30% |
| 08 | $\ln k \propto 1/T$ | $E_a$ | **0.02%** |
| 09 | $T \propto a^{3/2}$ | $GM_\odot$ | **0.04%** |
| 09 | same law, Jupiter's moons | $GM_{\rm Jup}$ | **0.02%** |
| 10 | $1/\lambda \propto 1/n^2$ | $R_H$ | **0.01%** |
| 11 | collective variable | — | ignored all decoys |

In none of these was the tool told what to look for. Notebook 1 sees 29 anonymous
columns and returns the area of a circle.

## Where it does not help

Three notebooks show a stock baseline matching or beating it, and the reason is the
same each time — **there was no multiplicative structure to find**:

| Notebook | Situation | Outcome |
|---|---|---|
| 01 | area is a degree-2 polynomial | ties `poly(2) + ridge` |
| 04 | penguin measurements near-linearly separable | logistic regression ties |
| 05 | diabetes has no compact algebraic law | ridge 0.393 vs beamfeat 0.371 |

That is the tool behaving correctly rather than failing. It is worth reaching for when
you suspect products and ratios matter; when you don't, a linear model is already
enough and `beamfeat` will roughly match it rather than blow up.

## The part that has no equivalent elsewhere

Other constructors will also find `x*y` if it is there. What none of them does is tell
you when there is nothing:

- **Pure noise** → zero features, `fdr_controlled_ = False`, explicit warning.
  `poly(2) + ridge` on the same data scored **−0.49** on held-out rows (notebook 5).
- **Distractor species** → zero false features across the stress suite (notebook 6).
- **13 rows** → `fdr_controlled_ = False` with a warning that the sample is too small
  to split (notebook 9).
- **Detection threshold** → nothing at SNR 0.1, 11/12 by SNR 0.4 (notebook 5).

## Reading order

Start with **07** (Reynolds number) or **06** (rate laws) if you want to see it work
on something real. Start with **05** if you are sceptical and want to see it refuse.
Start with **01** if you want the shortest path from raw columns to a known law.

| # | Notebook | Domain |
|---|---|---|
| 01 | Rediscovering geometry | shape measurements |
| 02 | Diamonds: volume from dimensions | 54,000 rows |
| 03 | Auto MPG: power-to-weight | engineering ratios, units |
| 04 | Palmer penguins | classification |
| 05 | Knowing when to stop | false discovery control |
| 06 | Reaction rate laws | chemical kinetics |
| 07 | Dimensionless groups | transport phenomena |
| 08 | Arrhenius activation energy | physical chemistry |
| 09 | Kepler's third law | celestial mechanics |
| 10 | Hydrogen spectrum | atomic physics |
| 11 | Collective variables | molecular dynamics |


## Three things that will bite you

Found while building this series, against `beamfeat 0.1.1`. Each fails **quietly**,
which is why they are here rather than in a footnote.

### 1. The variance floor is absolute, not relative

Columns with variance below `1e-10` are dropped as constant. Concentrations in
`kmol/m³` sit around `1e-6`, so their variance is `~1e-12`.

If *every* column falls below, you get a clear error. If only *some* do — trace
species alongside bulk ones, the normal case in chemistry — the fit succeeds with the
important column missing, and `fdr_controlled_` still reports `True`. Notebook 6
Part 3 demonstrates this: R² 0.795 and a completely wrong mechanism.

**Rescale inputs to roughly 0.01–100 and check `describe()` before fitting.**

### 2. `units=` as a list is silently ignored

Only the dict form, keyed by column name on a DataFrame, actually constrains the
search. The list form produces identical output with no error and no warning
(notebook 3).

**Always confirm the feature set changed before claiming dimensional validation.**

### 3. `equation()` drops small coefficients

Terms whose coefficient rounds to zero at four decimal places vanish from the printed
string, even though `predict()` uses them. Reproducer below.


In [1]:
%pip install -q "beamfeat[units]" pandas


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
import numpy as np
import pandas as pd
from beamfeat import BeamFeatRegressor

warnings.filterwarnings("ignore", message=".*valid feature names.*")

rng = np.random.default_rng(0)
T = rng.uniform(1000, 2000, 300)
y = 3e-7 * T ** 4 + 5000 + rng.normal(0, 1e4, 300)

m = BeamFeatRegressor(random_state=0).fit(pd.DataFrame({"T": T}), y)

print("true relationship :  y = 3.0e-07 * T^4 + 5000")
print("selected feature  :", m.formulas())
print("unscaled coef     :", m.coef_ / m.scaler_.scale_, "  <- correct")
print(f"R^2               : {m.score(pd.DataFrame({'T': T}), y):.5f}  <- model is fine")
print("equation()        :", m.equation(), "  <- term silently missing")


true relationship :  y = 3.0e-07 * T^4 + 5000
selected feature  : ['((T)^2)^2']
unscaled coef     : [2.98930509e-07]   <- correct
R^2               : 0.99994  <- model is fine
equation()        : y = 12395.7748   <- term silently missing


The model is right and `predict()` is right. Only the printed string is wrong, and
wrong by omission rather than rounding — which matters, because the readable equation
is the whole point of the tool.

**Read coefficients from `coef_ / scaler_.scale_`, not from `equation()`.**


## Setup for the whole series

Every notebook writes its data to a shared `csv/` folder, created on first run and
reused afterwards. Notebooks 2–4 download three public datasets once; everything else
generates its data and saves it. After one online run the series works offline.

```
pip install "beamfeat[units]" pandas matplotlib seaborn scikit-learn scipy cantera
```

`cantera` is only needed for notebook 6.
